In [1]:
import numpy as np

def doob_tilted_kernel(T, right_pf, rho):
    # T: (n,n) base kernel; rows sum to 1 if rho=1, else T is a linear operator with PF eigenvalue rho
    psi = right_pf
    W = (psi[None, :] / psi[:, None]) * T / rho
    # Row-normalize defensively for numerical safety
    W = W / W.sum(axis=1, keepdims=True)
    return W

def sample_chain(W, N, x0=0, rng=np.random.default_rng(0)):
    n = W.shape[0]
    cdf = np.cumsum(W, axis=1)
    x = x0
    traj = np.empty(N, dtype=int)
    for t in range(N):
        u = rng.random()
        x = np.searchsorted(cdf[x], u)
        traj[t] = x
    return traj

def autocorr(x, maxlag):
    x = (x - x.mean()) / (x.std() + 1e-12)
    ac = np.array([np.correlate(x[:-lag], x[lag:])[0]/(len(x)-lag) if lag>0 else 1.0 for lag in range(maxlag+1)])
    return ac

# Demo with a small reversible kernel
n = 50
rng = np.random.default_rng(1)
# Build a banded random walk with reflecting boundaries, then exponentiate to make a transfer operator
K = np.zeros((n,n))
for i in range(n):
    for j in [i-1, i, i+1]:
        if 0 <= j < n:
            K[i,j] = 1.0
K = K / K.sum(axis=1, keepdims=True)
# Treat K as T with rho=1 (Markov). PF right eigenvector is uniform; perturb to create a nontrivial ground state
vals, vecs = np.linalg.eig(K.T)            # left eigenvectors of K
idx = np.argmax(np.real(vals))
rho = np.real(vals[idx])
phi0 = np.real(vecs[:, idx]); phi0 = phi0/phi0.sum()
psi0 = np.ones(n)/n                         # right PF for stochastic K is uniform
# Create a shaped "ground state" by solving a Schrödinger-like tweak (toy): multiply by a bump and re-normalize
bump = np.exp(-((np.arange(n)-(n-1)/2)**2)/(2*(0.18*n)**2))
psi0 = psi0 * bump; psi0 = psi0/psi0.sum()

W = doob_tilted_kernel(K, psi0, rho=1.0)
traj = sample_chain(W, N=200000, x0=n//2, rng=rng)

# Measure an observable (position) and its autocorrelation
obs = traj.astype(float)
ac = autocorr(obs, maxlag=25)

print("First 6 lags of AC under Doob-tilted chain:", np.round(ac[:6], 4))
# A crude spectral-gap proxy from lag-1:
gap_proxy = -np.log(max(ac[1], 1e-12))
print("Effective gap proxy (lag-1):", gap_proxy)

First 6 lags of AC under Doob-tilted chain: [1.     0.9918 0.9837 0.9756 0.9676 0.9596]
Effective gap proxy (lag-1): 0.008186038612084771
